# 🎭 Emotive TTS — Master Colab Pipeline


**COMP3065 Project**

> **One-click setup:** This notebook clones the project from GitHub and runs
> entirely on Google Colab. Just click **Runtime → Run all**.

## Workflow
1. **Setup** — Mount Drive, clone repo from GitHub, install deps
2. **Data preparation** (S1) — Download & process EmoV-DB
3. **Training** — System A → B → C (S2-S4)
4. **Inference** — Generate eval stimuli (S5)
5. **Evaluation** — Prosody analysis + SER probe + plots (S6)

**Runtime:** GPU → T4 (Runtime → Change runtime type → T4 GPU)

---
**GitHub repo:** https://github.com/jimmy00415/COMP3067_Project

## 0. Setup

In [ ]:
# ── 0a. Mount Google Drive (for persistent checkpoint storage) ──
from google.colab import drive
drive.mount('/content/drive')

import os

# Persistent storage on Google Drive — survives runtime restarts
DRIVE_PROJECT = '/content/drive/MyDrive/COMP3065_Project'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/data/processed', exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_PROJECT}')

In [ ]:
# ── 0b. Clone project from GitHub ──
REPO_URL = 'https://github.com/jimmy00415/COMP3067_Project.git'
LOCAL_PROJECT = '/content/COMP3067_Project'

if os.path.exists(LOCAL_PROJECT):
    print('Repo already cloned — pulling latest...')
    !cd {LOCAL_PROJECT} && git pull
else:
    !git clone {REPO_URL} {LOCAL_PROJECT}

%cd {LOCAL_PROJECT}
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# 0c. GPU sanity check
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU     : {gpu_name}')
    print(f'VRAM    : {vram_gb:.1f} GB')
    if vram_gb < 10:
        print('⚠️  Less than 10 GB VRAM — training may OOM. Consider reducing batch size.')
else:
    print('⚠️  No GPU detected — training will be extremely slow!')

In [ ]:
# ── 0d. Install dependencies ──
import sys
print(f'Python version: {sys.version}')

# Coqui TTS 0.22.0 caps at Python <3.12, but Colab now ships 3.12+.
# Force-install with --ignore-requires-python (it works fine on 3.12).
!pip install -q TTS==0.22.0 --ignore-requires-python

# Core audio + ML infra
!pip install -q speechbrain>=1.0 librosa>=0.10
!pip install -q hydra-core>=1.3 omegaconf>=2.3 mlflow>=2.10
!pip install -q pandas matplotlib seaborn scipy scikit-learn soundfile
!pip install -q gradio>=4.0

# Install project in editable mode (makes `from src.xxx import ...` work)
!pip install -q -e .

# Verify installation
try:
    import TTS
    print(f'✅ Coqui TTS {TTS.__version__} installed')
except ImportError as e:
    print(f'❌ TTS import failed: {e}')

try:
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS
    from src.models.prosody_heads import build_prosody_heads
    from src.training.train import _compute_mel_spectrogram
    print(f'✅ Project modules OK. Emotions: {EMOTION_LABELS}')
except ImportError as e:
    print(f'❌ Project import failed: {e}\n   Re-run this cell or restart runtime.')

## 1. Data Preparation (S1)

In [ ]:
# ── 1a. Get EmoV-DB dataset ──
import os, tarfile, glob

EMOVDB_DRIVE = f'{DRIVE_PROJECT}/data/raw/EmoV-DB'
EMOVDB_LOCAL = 'data/raw/EmoV-DB'
os.makedirs('data/raw', exist_ok=True)

if os.path.exists(EMOVDB_DRIVE) and os.listdir(EMOVDB_DRIVE):
    # Symlink from Drive to avoid re-downloading
    if not os.path.exists(EMOVDB_LOCAL):
        os.symlink(EMOVDB_DRIVE, EMOVDB_LOCAL)
    print(f'✅ EmoV-DB found on Drive → symlinked to {EMOVDB_LOCAL}')

elif os.path.exists(EMOVDB_LOCAL) and os.listdir(EMOVDB_LOCAL):
    print(f'✅ EmoV-DB already at {EMOVDB_LOCAL}')

else:
    # Download from OpenSLR (SLR115) — per-speaker per-emotion tar.gz files
    print('⬇️  Downloading EmoV-DB from OpenSLR — this may take 10-15 min …')
    os.makedirs(EMOVDB_LOCAL, exist_ok=True)

    BASE_URL = 'https://www.openslr.org/resources/115'
    speakers = ['bea', 'jenie', 'josh', 'sam']
    # OpenSLR tar names → target emotion folder expected by prepare.py
    # Note: josh only has Amused/Neutral/Sleepy — missing entries are skipped
    emotions = {
        'Amused':    'Amused',
        'Angry':     'Angry',
        'Disgusted': 'Disgust',
        'Neutral':   'Neutral',
    }

    for emo_slr, emo_folder in emotions.items():
        emo_dir = os.path.join(EMOVDB_LOCAL, emo_folder)
        os.makedirs(emo_dir, exist_ok=True)

        for spk in speakers:
            fname = f'{spk}_{emo_slr}.tar.gz'
            tar_path = f'/tmp/{fname}'

            # Download (skip if already present)
            if not os.path.exists(tar_path) or os.path.getsize(tar_path) < 10_000:
                ret = os.system(f'wget -q --show-progress -O {tar_path} {BASE_URL}/{fname}')
                if ret != 0 or not os.path.exists(tar_path) or os.path.getsize(tar_path) < 10_000:
                    print(f'   ⏭️  {fname} not available, skipping')
                    continue

            # Extract — wav files go into the emotion folder
            try:
                print(f'   📦 {fname} → {emo_folder}/')
                with tarfile.open(tar_path, 'r:gz') as tf:
                    tf.extractall(emo_dir)
            except Exception as e:
                print(f'   ⚠️  Failed to extract {fname}: {e}')

    # Persist to Drive for future sessions
    print('\n   Copying raw data to Drive (for future sessions) …')
    !cp -r {EMOVDB_LOCAL} {DRIVE_PROJECT}/data/raw/ 2>/dev/null || true
    print(f'✅ EmoV-DB downloaded and extracted to {EMOVDB_LOCAL}')

# Verify what we have
subdirs = sorted(d for d in os.listdir(EMOVDB_LOCAL)
                 if os.path.isdir(os.path.join(EMOVDB_LOCAL, d)))
print(f'\n   Emotion folders: {subdirs}')
for d in subdirs:
    wavs = glob.glob(os.path.join(EMOVDB_LOCAL, d, '**', '*.wav'), recursive=True)
    print(f'   {d}: {len(wavs)} wav files')

In [ ]:
# ── 1b. Run data preparation pipeline ──
import os, yaml
from src.data.prepare import prepare_dataset

with open('configs/data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f)

# Override raw_dir for Colab path
data_cfg['dataset']['raw_dir'] = EMOVDB_LOCAL

# Sanity check
assert os.path.isdir(EMOVDB_LOCAL), (
    f"EmoV-DB not found at {EMOVDB_LOCAL}. Re-run cell 1a first."
)

summary = prepare_dataset(data_cfg)
print('\n📊 Dataset summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Persist manifests to Drive
!cp -r data/manifests {DRIVE_PROJECT}/data/ 2>/dev/null || true
print(f'\n✅ Manifests saved to Drive')

In [ ]:
# ── 1c. Run Data QA ──
from src.data.qa import generate_qa_report

generate_qa_report(
    manifest_path='data/manifests/full_manifest.csv',
    output_dir='figures/data_qa',
    report_path='docs/data_qa_report.md',
)
print('✅ QA report: docs/data_qa_report.md')

# Quick preview
from IPython.display import Markdown, display
if os.path.exists('docs/data_qa_report.md'):
    with open('docs/data_qa_report.md') as f:
        display(Markdown(f.read()[:2000] + '\n\n*... (truncated)*'))

## 2. Training: System A (S2) — Domain adaptation

System A fine-tunes pretrained VITS on EmoV-DB **without** emotion labels.
This is the domain-adapted baseline to isolate the conditioning effect (A→B).

In [ ]:
# ── 2a. Train System A ──
import yaml, torch
from src.training.train import train

with open('configs/train_a.yaml', 'r') as f:
    train_a_cfg = yaml.safe_load(f)

# Colab overrides
train_a_cfg['use_cuda'] = True
train_a_cfg.setdefault('training', {})['fp16'] = True

# Restore from Drive checkpoint if available
drive_ckpt = f'{DRIVE_PROJECT}/checkpoints/system_a/best.pth'
if os.path.exists(drive_ckpt):
    print(f'📂 Restoring System A checkpoint from Drive: {drive_ckpt}')
    train_a_cfg['init_from'] = drive_ckpt

print('🚀 Starting System A training...')
if torch.cuda.is_available():
    print(f'   VRAM before: {torch.cuda.memory_allocated()/1e9:.2f} GB')
result_a = train(train_a_cfg)
print('✅ System A training complete:', result_a)

In [ ]:
# ── 2b. Save System A checkpoint to Drive ──
!mkdir -p {DRIVE_PROJECT}/checkpoints/system_a
!cp -r checkpoints/system_a/* {DRIVE_PROJECT}/checkpoints/system_a/ 2>/dev/null
print('✅ System A checkpoint saved to Drive')

## 3. Training: System B (S3) — Emotion embedding

System B = System A + `nn.Embedding(4, 192)` for emotion conditioning.
Initialises from the **System A checkpoint** (warm start).

In [ ]:
# ── 3a. Train System B ──
with open('configs/train_b.yaml', 'r') as f:
    train_b_cfg = yaml.safe_load(f)

train_b_cfg['use_cuda'] = True
train_b_cfg.setdefault('training', {})['fp16'] = True

# Warm-start from System A
drive_ckpt_a = f'{DRIVE_PROJECT}/checkpoints/system_a/best.pth'
local_ckpt_a = 'checkpoints/system_a/best.pth'
if os.path.exists(drive_ckpt_a):
    train_b_cfg['init_from'] = drive_ckpt_a
elif os.path.exists(local_ckpt_a):
    train_b_cfg['init_from'] = local_ckpt_a

print('🚀 Starting System B training...')
result_b = train(train_b_cfg)
print('✅ System B training complete:', result_b)

In [ ]:
# ── 3b. Save System B checkpoint to Drive ──
!mkdir -p {DRIVE_PROJECT}/checkpoints/system_b
!cp -r checkpoints/system_b/* {DRIVE_PROJECT}/checkpoints/system_b/ 2>/dev/null
print('✅ System B checkpoint saved to Drive')

## 4. Training: System C (S4) — Prosody supervision

System C = System B + utterance-level prosody auxiliary heads (F0 + energy).
Initialises from the **System B checkpoint** (warm start). Loss weight λ = 0.1.

In [ ]:
# ── 4a. Train System C ──
with open('configs/train_c.yaml', 'r') as f:
    train_c_cfg = yaml.safe_load(f)

train_c_cfg['use_cuda'] = True
train_c_cfg.setdefault('training', {})['fp16'] = True

# Warm-start from System B
drive_ckpt_b = f'{DRIVE_PROJECT}/checkpoints/system_b/best.pth'
local_ckpt_b = 'checkpoints/system_b/best.pth'
if os.path.exists(drive_ckpt_b):
    train_c_cfg['init_from'] = drive_ckpt_b
elif os.path.exists(local_ckpt_b):
    train_c_cfg['init_from'] = local_ckpt_b

print('🚀 Starting System C training...')
result_c = train(train_c_cfg)
print('✅ System C training complete:', result_c)

In [ ]:
# ── 4b. Save System C checkpoint to Drive ──
!mkdir -p {DRIVE_PROJECT}/checkpoints/system_c
!cp -r checkpoints/system_c/* {DRIVE_PROJECT}/checkpoints/system_c/ 2>/dev/null
print('✅ System C checkpoint saved to Drive')

# Clear GPU cache before inference
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'   VRAM after clearing: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 5. Inference (S5)

Generate evaluation stimuli: **16 canary texts × 4 emotions × 4 systems = 256 samples**

In [ ]:
# 5a. Generate evaluation stimuli for all 4 systems
import yaml, os, torch
from src.inference.run import run_inference

with open('configs/infer.yaml', 'r') as f:
    infer_cfg = yaml.safe_load(f)

# Force CUDA on Colab
infer_cfg['inference']['use_cuda'] = torch.cuda.is_available()
# Ensure all 4 systems are generated for evaluation
infer_cfg['inference']['systems'] = ['A0', 'A', 'B', 'C']
# Ensure output goes where evaluation expects it
infer_cfg['inference']['output_dir'] = 'data/processed/eval_stimuli'

print("🎙️ Running inference — generating evaluation samples for all 4 systems …")
results = run_inference(infer_cfg)

n_files = results.get('total_files', 0)
print(f"✅ Inference complete — {n_files} files generated")
print(f"   Systems: {results.get('systems', [])}")
print(f"   Manifest: {results.get('manifest_path', 'N/A')}")

In [ ]:
# 5b. Persist eval stimuli to Google Drive
import os, shutil

drive_eval = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'eval_stimuli')
os.makedirs(drive_eval, exist_ok=True)
shutil.copytree('data/processed/eval_stimuli', drive_eval, dirs_exist_ok=True)
print(f"💾 Eval stimuli saved → {drive_eval}")

## 6. Evaluation (S6)

Run prosody analysis (PRIMARY), SER probe (AUXILIARY), generate plots, and create listening-test stimulus pack.

In [ ]:
# 6a. Prosody analysis (PRIMARY metric — F0 & energy stats)
import yaml
from src.evaluation.prosody import run_prosody_evaluation

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print("📊 Running prosody evaluation …")
prosody_results = run_prosody_evaluation(eval_cfg)
print("✅ Prosody evaluation complete")

# Show aggregate stats if available
if 'agg_stats' in prosody_results:
    from IPython.display import display
    display(prosody_results['agg_stats'])
else:
    for k, v in prosody_results.items():
        print(f"  {k}: {v}")

In [ ]:
# 6b. SER probe (AUXILIARY metric — label mismatch caveat applies)
from src.evaluation.ser_probe import run_ser_evaluation

print("🧠 Running SER probe evaluation …")
ser_results = run_ser_evaluation(eval_cfg)

agreement = ser_results.get('ser_proxy_agreement', 0)
n_excluded = ser_results.get('n_excluded', 0)
print(f"✅ SER proxy agreement: {agreement:.2%}")
print(f"   ⚠️  This is an AUXILIARY metric only (Wav2Vec2-based proxy).")
if n_excluded > 0:
    print(f"   Excluded {n_excluded} samples with unmapped emotions (e.g. disgust).")

In [ ]:
# 6c. Generate all plots
import os
from src.evaluation.plots import generate_all_plots

print("📈 Generating plots …")
generate_all_plots(eval_cfg)
print("✅ All plots saved to figures/")

# Display key plots inline
from IPython.display import Image, display as ipy_display
for plot_name in ['f0_by_system_emotion.png', 'prosody_comparison_grid.png']:
    plot_path = f'figures/{plot_name}'
    if os.path.exists(plot_path):
        print(f"\n--- {plot_name} ---")
        ipy_display(Image(filename=plot_path, width=800))
    else:
        print(f"   (plot not found: {plot_name})")

In [ ]:
# 6d. Generate listening test stimulus pack
from src.evaluation.listening_test import create_stimulus_pack

print("🎧 Creating listening test stimulus pack …")
stim_summary = create_stimulus_pack(
    manifest_path='data/processed/eval_stimuli/eval_manifest.csv',
    output_dir='outputs/listening_test',
)
print(f"✅ Stimulus pack ready: {stim_summary}")

In [ ]:
# 6e. Save ALL outputs to Google Drive
import os, shutil

for folder in ['tables', 'figures', 'outputs', 'docs']:
    src_dir = folder
    dst_dir = os.path.join(DRIVE_PROJECT, folder)
    if os.path.isdir(src_dir):
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"  💾 {folder}/ → Drive")
    else:
        print(f"  ⏭️  {folder}/ not found, skipping")

print("\n✅ All outputs saved to Google Drive")

## 7. Summary

| Step | Section | Status |
|------|---------|--------|
| Data prep | S1 | ✅ |
| System A (baseline VITS) | S2 | ✅ |
| System B (+ emotion embedding) | S3 | ✅ |
| System C (+ prosody heads, λ=0.1) | S4 | ✅ |
| Inference (256 samples) | S5 | ✅ |
| Prosody eval (PRIMARY) | S6a | ✅ |
| SER probe (AUXILIARY) | S6b | ✅ |
| Plots | S6c | ✅ |
| Listening test pack | S6d | ✅ |

**All checkpoints and outputs are persisted to Google Drive** under:
`/content/drive/MyDrive/COMP3065_Project/`

> **Tip:** If the Colab session disconnects mid-training, re-run from the top —
> Drive checkpoint restore will skip completed training stages automatically.